# Image Feature Extraction

This notebook follows along with the slides on **Image Feature Extraction** Lecture 33: Image Feature Extraction, using the `machinevisiontoolbox` package.


Note: Code cahnges in this notebook from the lecture are:
- For compatibility with NumPy 2.x, np.vstack replaces the deprecated/removed np.row_stack used internally by the Hough plotting code.
- plot_point() is no longer available as a standalone function, so use Matplotlib’s ax.plot() to overlay points on the chromaticity diagram.

## Environment Setup

Detect whether the notebook is running on Google Colab (installing `matplotlib` and `machinevision-toolbox-python` if so), then import `numpy`, `matplotlib`, and the Machine Vision Toolbox (`machinevisiontoolbox`), which provides the `Image` class and feature-extraction utilities used throughout this notebook.

In [ ]:
try:
    import google.colab
    print('Running on CoLab')
    !pip install matplotlib
    !pip install machinevision-toolbox-python
    COLAB = True
except ModuleNotFoundError:
    COLAB = False

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
np.set_printoptions(
    linewidth=120, formatter={
        'float': lambda x: f"{0:8.4g}" if abs(x) < 1e-10 else f"{x:8.4g}"})
np.random.seed(0)

from machinevisiontoolbox import *
from machinevisiontoolbox import plot_chromaticity_diagram
from matplotlib.patches import Ellipse

import torch
import torchvision as tv

## 1. Region Features

Region features include 3 stages: **pixel classification**, **instance segmentation**, and **description**.

### 1.1 Pixel Classification
Pixel Classification for Monochrome Images:

- **Thresholding**: pixels are classified into 2 classes, either object ($c=1$) or not-object ($c=0$) based on a threshold.
- **Otsu's Method**: an automated thresholding technique that computes the optimal threshold by minimizing intra-class variance and maximizing inter-class variance.

In [ ]:
castle = Image.Read("castle.png", dtype="float")
castle.disp()

t = 0.7               # Manual Threshold
t = castle.otsu()     # Compute Otsu Threshold
(castle >= t).disp();

- **Adaptive Thresholding**: a single threshold fails in real-world scenes with uneven lighting. A unique threshold is computed for every pixel based on a local window $W_{u,v}$ around $(u,v)$:
$$t_{u,v} = \mu(W_{u,v}) - k\,\sigma(W_{u,v})$$
where $\mu, \sigma$ are the mean and standard deviation of the window, and $k$ is an offset parameter.

In [ ]:
# adaptive Threshold with window size=15
castle.threshold_adaptive(h=15).disp();

Pixel Classification for Color Images - Strategies:

- **Thresholding specific planes** $a^*$ (green to red), $b^*$ (blue to yellow) after discarding the $L^*$ (luminance) plane.
- **k-Means Clustering** (threshold-free approach): an unsupervised learning technique that groups pixels into $C$ classes based on their similarity in the $a^*b^*$ chromaticity space.

We use four yellow markers (from an indoor drone landing experiment) and find the centroids of the yellow targets using k-means.

In [ ]:
targets = Image.Read("yellowtargets.png", dtype="float", gamma="sRGB")

# Convert to L*a*b* space and extract color planes (a* and b*)
ab = targets.colorspace("L*a*b*").plane("a*:b*")
ab.plane("b*:").disp()   # plot b* plane

In [ ]:
# Cluster chromaticity data into 2 classes, seed=0 ensures consistent results
targets_labels, targets_centroids, resid = ab.kmeans_color(k=2, seed=0)
targets_labels.disp(colormap="jet", colorbar=True);

The pixels are clustered into two chromaticity classes {0, 1}, i.e. yellow targets and a gray background — the cluster closest to yellow is displayed in false colour.

In [ ]:
chrom = plot_chromaticity_diagram(colorspace="a*b*")
plt.imshow(chrom, extent=(-128, 127, -128, 127), origin="lower")

plt.plot(targets_centroids[:, 0], targets_centroids[:, 1], "*")

for i, (a, b) in enumerate(targets_centroids):
    plt.text(a, b, str(i), fontsize=12)

plt.show()

### 1.2 Instance Segmentation

This stage transforms the segmented pixel groups (blobs) into structured representations that a robot can use for decision-making, by labelling pixels to spatial sets for $m$ distinct objects as $c = 0, 1, \dots m$. 

In [ ]:
targets_labels_uint8 = targets_labels.astype(np.uint8)   # convert to uint8 data type
target_labels_scaled = (1 - targets_labels_uint8) * 255  # scaling to 0-255
target_labels_scaled.disp()

In [ ]:
labels, m = target_labels_scaled.labels_binary()  # apply label to each region
labels.disp(colorbar=True);

top_box = (labels == 1)     # extract top box with index 1
top_box.disp()
background = (labels == 0)  # extract background with index 0

### 1.3 Description

Description reduces a region (potentially thousands of pixels) into a few compact scalar or vector features.

#### Shape from Moments

Moments provide a computationally cheap way to describe the size, location, orientation, and shape of regions. **Area** ($m_{00}$); **Centroid** $u_c = m_{10}/m_{00}$ and $v_c = m_{01}/m_{00}$; **Equivalent Ellipse**: eigenvalues of the inertia tensor $J = \begin{bmatrix}\mu_{20} & \mu_{11}\\ \mu_{11} & \mu_{02}\end{bmatrix}$

In [ ]:
blobs = target_labels_scaled.blobs()  # Description of the blobs
blobs

In [ ]:
# descriptors of any blob id (here id=3, top box)
a3 = blobs[3].area                 # area of the blob
umin3 = blobs[3].umin              # min value of u
asp3 = blobs[3].aspect             # aspect ratio
(uc3, vc3) = blobs[3].centroid     # centroid
blobs.humoments()                  # hu moments of all blobs

m00 = blobs[3].moments.m00         # 0th moment ~ .area (=a3)
u20 = blobs[3].moments.mu20        # central moment p=2 q=0
J3 = np.array([[u20, blobs[3].moments.mu11],
               [blobs[3].moments.mu11, blobs[3].moments.mu02]])  # inertia tensor

print(a3, umin3, asp3, (uc3, vc3))

In [ ]:
target_labels_scaled.disp(block=None)              # blobs
blobs.plot_centroid(marker="+", color="blue")      # centroids
blobs.plot_box(color="red")                        # bounding box

# plot_ellipse(4 * J3 / m00, centre=(uc3, vc3), inverted=True, color="green");  # plot ellipse

M = 4 * J3 / m00
eigval, eigvec = np.linalg.eigh(M)
angle = np.degrees(np.arctan2(eigvec[1, 1], eigvec[0, 1]))

plt.gca().add_patch(
    Ellipse((uc3, vc3),
            2*np.sqrt(eigval[1]),
            2*np.sqrt(eigval[0]),
            angle=angle,
            fill=False,
            color="green")
)
plt.show()

#### Shape from Perimeter

A region can also be described by its edgels (perimeter, boundary, or contour pixels): Chain Codes, Circularity and Radius Signature

In [ ]:
# Get the radius (r) and angle (th) to every point
# on the perimeter of id=0
r, th = blobs[0].polar()

# Compare the shape of blob 1 with all others in the scene
# (1.0 = perfect match)
similarity, _ = blobs.polarmatch(1)

plt.plot(r, "r", th, "b");  # plot the radius and angle

In [ ]:
for blob in blobs:      # radius plot of all blobs
    r, theta = blob.polar()
    plt.plot(r / r.sum())

In [ ]:
blobs[3].perimeter.shape      # no. of perimeter pixels
p = blobs[3].perimeter_length  # perimeter length

blobs[3].plot_perimeter(color="orange")  # plot perimeter
print(blobs.circularity)                 # circularity

### Object Detection using Deep Learning

Modern robotic systems often bypass manual feature engineering by using deep networks like **Faster R-CNN**, trained on the COCO dataset (80 object classes including person, bicycle, car, cat, dog, etc.). These networks perform pixel classification, instance segmentation, and description (generating a bounding box and class label) in a single integrated pipeline.

In [ ]:
scene = Image.Read("image3.jpg")
scene.disp();

transform = tv.transforms.Compose([
    tv.transforms.ToTensor(),
    tv.transforms.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225])
])

in_tensor = transform(scene.image)

model = tv.models.segmentation.fcn_resnet50(pretrained=True).eval()
outputs = model(torch.stack([in_tensor]));

labels = Image(torch.argmax(outputs["out"].squeeze(),
                             dim=0).detach().cpu().numpy());

labels.disp(colormap="viridis", ncolors=20, colorbar=True);

(labels == 15).disp();
scene.choose("white", labels != 15).disp();

## 2. Line Features

In contrast to region features, which look at contiguous groups of pixels, **line features** focus on the geometric structure of a scene. Straight lines are particularly prevalent in human-made environments — the edges of buildings, roads, and doorways.

### 2.1 Geometric Line Fitting - Hough Transform

In [ ]:
points5 = Image.Read("5points.png", mono="True")

h = points5.Hough()    # Compute the Hough transform
h.plot_accumulator()   # accumulator array
plt.colorbar()

In [ ]:
np.row_stack = np.vstack

lines = h.lines(2)    # dominant lines with min votes=3
line = lines[0]        # select the first detected line
points5.disp()
h.plot_lines(np.array([line]));  # plot lines

### 2.2 Canny Edges

The **Canny edge detector** is the most widely used edge detector in robotics — a multi-stage algorithm. The **Hough transform** is used to detect straight lines and other parametric shapes: it converts points in the image into votes in a parameter space, and peaks in that space indicate likely shapes in the original image.

In [ ]:
import cv2 as cv

church = Image.Read("church.png", mono=True)
img = church.to_int()
# gaussian smoothing with 5x5 kernel & std=1.4
smooth = cv.GaussianBlur(img, (5, 5), 1.4)

edges = cv.Canny(smooth, 50, 100)
edges_img = Image(edges)
edges_img.disp()

In [ ]:
h = edges_img.Hough()
lines = h.lines_p(100, minlinelength=200, maxlinegap=5, seed=0)
church.disp()
h.plot_lines(lines);

## 3. Point Features

Point features are visually distinct locations in an image that can be reliably detected across different views of the same scene. They are critical for robotic tasks such as stereo vision, motion estimation, and object recognition, since they are robust to illumination, rotation, and scale changes.

### 3.1 Harris Corner Detector

A robust corner detector that uses the auto-correlation matrix

In [ ]:
view1 = Image.Read("building2-1.png", mono=True)
view1.disp()

In [ ]:
# nfeat: limits the output to the most keypoints
harris1 = view1.Harris(nfeat=500)
view1.disp(darken=True)
harris1.plot()

### 3.2 Scale-Invariant Point Features and SIFT

In [ ]:
# retains only top 50% strength keypoints and larger, stable structural landmarks (scale >= 5 pixels)
sift1 = view1.SIFT().filter(percentstrength=50, minscale=5)

sift1[0]  # Access properties of the first SIFT feature

# plot draws a circle representing the scale, hand=True draws a radial line of dominant orientation
view1.disp(block=None, darken=True)
sift1.plot(filled=True, color="y", hand=True, alpha=0.3)

### Try it yourself
- Try Otsu's threshold vs. a manual threshold vs. adaptive thresholding on a different image with uneven illumination and compare the results.
- Change `k` in `kmeans_color` to segment the color image into more than 2 classes.
- Adjust the Harris `nfeat` or the SIFT `percentstrength`/`minscale` filters and observe how the number and stability of detected keypoints changes.